Takes the refrences transformed to json using anystyle and puts them to the GEMS SDRef bibliographic format. Checks the refrences excel file and uses they key there to update the key in the json record

In [1]:
import json
import pandas as pd
from fuzzywuzzy import process  # or: from rapidfuzz import process

# ------------------------------------------------------------
# Load Excel mapping
# ------------------------------------------------------------
def load_reference_keys(file_path):
    df = pd.read_excel(file_path, sheet_name="Sheet1")
    return df[["Ref_full", "Ref_GEMS"]].dropna()


# ------------------------------------------------------------
# Fuzzy match title → Ref_GEMS
# ------------------------------------------------------------
def find_partial_match(title, reference_df, threshold=80):
    matches = process.extractOne(title, reference_df["Ref_full"], score_cutoff=threshold)
    if matches:
        matched_title, score, index = matches
        return reference_df.loc[index, "Ref_GEMS"]
    print("NOT FOUND:", title)
    return ""

def format_authors(authors):
    """Format authors as 'A', 'A and B', or 'A, B, C and D'."""
    if not authors:
        return ""

    names = [
        f"{a.get('given', '')} {a.get('family', '')}".strip()
        for a in authors
    ]

    if len(names) == 1:
        return names[0]

    if len(names) == 2:
        return f"{names[0]} and {names[1]}"

    # 3 or more authors
    return ", ".join(names[:-1]) + f" and {names[-1]}"


# ------------------------------------------------------------
# Convert CSL JSON → SDref.backup.json structure
# ------------------------------------------------------------
def convert_to_sdref(bib_data, reference_df):
    formatted = []

    for entry in bib_data:
        # --- Authors ---------------------------------------------------------
        authors = entry.get("author", [])
        author_list = [
            f"{a.get('given', '')} {a.get('family', '')}".strip()
            for a in authors
        ]
        SDauth = format_authors(authors)

        # --- Title -----------------------------------------------------------
        title = entry.get("title", "")

        # --- Match to Ref_GEMS ----------------------------------------------
        ref_key = find_partial_match(title, reference_df)
        parts = ref_key.split(":") if ref_key else ["", "", ""]
        author_key = parts[0] if len(parts) > 0 else ""
        year_key = parts[1] if len(parts) > 1 else ""
        type_key = parts[2] if len(parts) > 2 else ""

        # --- Editor / container / publisher ---------------------------------
        SDedit = (
            (entry.get("genre", "") + " " +
             entry.get("container-title", entry.get("publisher", "")))
            .strip()
        )

        # --- Volume + year ---------------------------------------------------
        SDvoly = f"{year_key}, {entry.get('volume', '')}"

        # --- Page ------------------------------------------------------------
        SDpage = entry.get("page", "")

        # --- Build SDref.backup.json entry ----------------------------------
        formatted_entry = {
            "dod": {
                "SD_kwd": None,
                "SDabst": "---",
                "SDauth": SDauth,
                "SDedit": SDedit,
                "SDnote": "---",
                "SDpage": SDpage,
                "SDrefs": [[" "]],
                "SDtitl": title,
                "SDvoly": SDvoly
            },
            "key": [
                author_key,
                year_key,
                type_key
            ],
            "project": "backup"
        }

        formatted.append(formatted_entry)
        print("→", author_key)

    return formatted


# ------------------------------------------------------------
# Save JSON
# ------------------------------------------------------------
def save_json(data, filename="SDref.generated.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)


In [2]:
# Load reference keys
reference_file = "references.xlsx"
reference_keys = load_reference_keys(reference_file)

# Read and process bibliographic JSON data
bibliographic_data = json.load(open("references.json", "r", encoding="utf-8"))
converted_data = convert_to_sdref(bibliographic_data, reference_keys)

# Print the output in JSON format
#print(json.dumps(converted_data, indent=4))
# Save formatted data to a JSON file
save_json(converted_data)

print(f"Formatted data has been saved to 'formatted_references.json'.")

→ Cox_ea
→ Brown_ea
→ Hummel_ea
→ Hummel_ea
→ Hummel_ea
→ Lemire_ea
→ Lemire_ea
→ Rand_ea
→ Miron
→ Tanger_ea
→ Shock_ea
→ Shock_ea
→ Robie_ea
→ Shen_ea
→ Blanc_ea
→ Ma_ea
→ Ma_ea_b
→ Ma_ea
→ Kulik
→ Lothenbach_ea
→ Myers_ea
→ Nied_ea
→ Kulik_ea
→ Miron_ea
→ Miron_ea_b
→ Li_ea
→ Miron_b
→ Chase
→ Wolery
→ Miron
→ Gurvich_ea
→ Gurvich_ea
→ Olin_ea
→ Lothenbach_ea
→ Lothenbach_ea
Formatted data has been saved to 'formatted_references.json'.
